# Six Degrees of MoMA: 03 GNNs and the curator ablation

This notebook trains encoder–decoder link predictors on rolling temporal snapshots and evaluates
them with the same protocol as the baselines.

- **Heterogeneous** (`rgcn`, `heteroconv`, `hgt`): artist, exhibition and curator nodes.
  Test exhibitions enter the graph with their metadata and curator edges only.
- **Homogeneous** (`sage`, `gat`): the artist co-exhibition graph. Exhibitions are embedded from
  metadata with an MLP.
- **Curator modes**: `full`; `none` (curator node type dropped); `shuffled` (curator→exhibition
  assignments permuted within each decade, keeping all degrees).

Training runs on the GPU when one is available (Colab: Runtime → Change runtime type → GPU).
Outputs go to `results/gnn/`. Run `02_baselines.ipynb` first to include baselines in the final
comparison.

In [ ]:
# Installs anything missing. Colab already has torch, pandas, scikit-learn and matplotlib.
import importlib.util, subprocess, sys

REQUIRED = {"torch": "torch", "torch_geometric": "torch_geometric", "pyarrow": "pyarrow",
            "sentence_transformers": "sentence-transformers", "scipy": "scipy",
            "sklearn": "scikit-learn", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
from pathlib import Path

# On Colab, set USE_DRIVE = True to keep data and results across sessions.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/moma-project")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
for d in [RAW_DIR, PROCESSED_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    # Raw ExhibitionRole values mapped onto node roles; everything else is "other".
    # Institutions listed with a curator role are treated as "other" (organizers).
    "roles": {"artist": ["Artist"], "curator": ["Curator"]},
    "features": {"title_encoder": "minilm",   # minilm | tfidf
                 "title_dim": 32, "top_nationalities": 12},
    "split": {
        "cutoffs": [1950, 1960, 1970, 1980],
        "val_years": 5,       # val targets:  [T - val_years, T)
        "test_years": 10,     # test targets: [T, T + test_years)
        "train_window": 5,    # rolling training targets: [t, t + train_window)
        "train_start": 1932,  # earliest pseudo-cutoff for training snapshots
    },
    "eval": {"ks": [10, 20, 50], "primary_metric": "map@20"},
    "baselines": {
        "recency_tau": 10.0,
        "node2vec": {"dim": 64, "walk_length": 20, "walks_per_node": 10, "window": 4,
                     "p": 1.0, "q": 1.0, "epochs": 2, "num_neg": 5},
    },
    "model": {
        "name": "rgcn",           # rgcn | heteroconv | hgt | sage | gat
        "curator_mode": "full",   # full | none | shuffled
        "hidden": 64, "layers": 2, "dropout": 0.2, "lr": 0.005, "weight_decay": 1e-4,
        "epochs": 300, "eval_every": 5, "patience": 10,   # patience in evaluation rounds
        "loss": "bpr",            # bpr | bce
        "num_neg": 20,
        "max_group": None,        # subsample group shows above this size (homogeneous graph)
    },
    "seeds": [0, 1, 2, 3, 4],
}

## Core: data loading and cleaning

The CSV has one row per (exhibition, constituent, role) and is cp1252-encoded.

- **Exhibitions** are keyed on (number, title, begin date). ExhibitionID is missing for about 430 rows, and
  the number "No#" covers about 50 different exhibitions. `exh_id` is the chronological index.
- **Constituents** without a ConstituentID are keyed by display name.
- **Roles**: `artist`, `curator` (an individual with `ExhibitionRole == "Curator"`) or `other`.

In [ ]:
import copy, json, time, urllib.request
from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "https://raw.githubusercontent.com/MuseumofModernArt/exhibitions/master/"
EXHIBITIONS_FILE = "MoMAExhibitions1929to1989.csv"
TABLES = ["exhibitions", "constituents", "participation"]


def download(raw_dir, force=False):
    dest = raw_dir / EXHIBITIONS_FILE
    if force or not dest.exists():
        print(f"downloading {EXHIBITIONS_FILE}")
        urllib.request.urlretrieve(BASE_URL + EXHIBITIONS_FILE, dest)
    return dest


def load_raw(raw_dir):
    return pd.read_csv(raw_dir / EXHIBITIONS_FILE, encoding="cp1252", dtype=str)


def build_exhibitions(raw):
    ex = raw.drop_duplicates("exh_key")[[
        "exh_key", "ExhibitionID", "ExhibitionNumber", "ExhibitionTitle",
        "ExhibitionBeginDate", "ExhibitionEndDate", "ExhibitionSortOrder", "ExhibitionURL",
    ]].copy()
    ex["begin"] = pd.to_datetime(ex["ExhibitionBeginDate"], format="%m/%d/%Y")
    ex["end"] = pd.to_datetime(ex["ExhibitionEndDate"], format="%m/%d/%Y")
    ex["sort_order"] = pd.to_numeric(ex["ExhibitionSortOrder"], errors="coerce")
    ex = ex.sort_values(["begin", "sort_order", "exh_key"]).reset_index(drop=True)
    ex["exh_id"] = np.arange(len(ex))
    ex["year"] = ex["begin"].dt.year + (ex["begin"].dt.day_of_year - 1) / 365.25
    duration = (ex["end"] - ex["begin"]).dt.days
    ex["duration_days"] = duration.where(duration >= 0)
    return ex.rename(columns={
        "ExhibitionID": "source_id", "ExhibitionNumber": "number",
        "ExhibitionTitle": "title", "ExhibitionURL": "url",
    })[["exh_id", "exh_key", "source_id", "number", "title", "begin", "end", "year",
        "duration_days", "url"]]


def build_constituents(rows):
    def first_valid(s):
        s = s.dropna()
        return s.iloc[0] if len(s) else None

    cols = {
        "DisplayName": "name", "ConstituentType": "type", "Nationality": "nationality",
        "Gender": "gender", "ConstituentBeginDate": "birth_year",
        "ConstituentEndDate": "death_year", "WikidataID": "wikidata", "ULANID": "ulan",
    }
    con = rows.groupby("cid")[list(cols)].agg(first_valid).rename(columns=cols).reset_index()
    con["gender"] = con["gender"].str.lower().fillna("unknown")
    con["nationality"] = con["nationality"].fillna("unknown")
    con["type"] = con["type"].fillna("unknown")
    for col in ["birth_year", "death_year"]:
        year = pd.to_numeric(con[col], errors="coerce")
        con[col] = year.where(year > 0)
    return con


def build_participation(rows, roles):
    raw_role = rows["ExhibitionRole"]
    role = pd.Series("other", index=rows.index)
    role[raw_role.isin(roles["artist"])] = "artist"
    is_curator = raw_role.isin(roles["curator"]) & (rows["ConstituentType"] != "Institution")
    role[is_curator] = "curator"
    part = pd.DataFrame({
        "exh_id": rows["exh_id"].values,
        "cid": rows["cid"].values,
        "role": role.values,
        "raw_role": raw_role.values,
        "press_role": rows["ExhibitionRoleinPressRelease"].values,
    })
    return part.drop_duplicates(["exh_id", "cid", "role"]).reset_index(drop=True)


def clean(raw, roles):
    raw = raw.copy()
    raw["exh_key"] = (raw["ExhibitionNumber"].fillna("") + "|" + raw["ExhibitionTitle"].fillna("")
                      + "|" + raw["ExhibitionBeginDate"].fillna(""))
    exhibitions = build_exhibitions(raw)
    raw["exh_id"] = raw["exh_key"].map(exhibitions.set_index("exh_key")["exh_id"])

    # Rows with no role and no constituent only record that an exhibition happened.
    rows = raw[raw["ExhibitionRole"].notna() & raw["DisplayName"].notna()].copy()
    rows["cid"] = ("moma:" + rows["ConstituentID"]).fillna("name:" + rows["DisplayName"].str.strip())
    constituents = build_constituents(rows)
    participation = build_participation(rows, roles)

    has_curator = participation.loc[participation["role"] == "curator", "exh_id"].unique()
    exhibitions["has_curator"] = exhibitions["exh_id"].isin(has_curator)
    return {"exhibitions": exhibitions, "constituents": constituents,
            "participation": participation}


def load_tables(force=False):
    """Cleaned tables from data/processed, rebuilding them from the raw CSV if needed."""
    if not force and all((PROCESSED_DIR / f"{n}.parquet").exists() for n in TABLES):
        return {n: pd.read_parquet(PROCESSED_DIR / f"{n}.parquet") for n in TABLES}
    download(RAW_DIR)
    tables = clean(load_raw(RAW_DIR), CFG["roles"])
    for name, df in tables.items():
        df.to_parquet(PROCESSED_DIR / f"{name}.parquet", index=False)
    return tables

### Integer-indexed dataset

Artists and curators have separate node index spaces. A person who both exhibited and curated
appears once in each; both keep `cid` so they can be merged back into one person when needed.

`with_shuffled_curators` is the shuffle ablation. It permutes curator→exhibition assignments
within each decade, so every curator keeps their number of exhibitions and every exhibition
keeps its number of curators.

In [ ]:
@dataclass
class MomaData:
    exhibitions: pd.DataFrame    # indexed by exh_id (chronological)
    artists: pd.DataFrame        # indexed by artist node id
    curators: pd.DataFrame       # indexed by curator node id
    artist_edges: pd.DataFrame   # columns exh_id, artist
    curator_edges: pd.DataFrame  # columns exh_id, curator

    @classmethod
    def from_tables(cls, tables):
        ex = tables["exhibitions"].set_index("exh_id", drop=False).sort_index()
        con = tables["constituents"].set_index("cid")
        part = tables["participation"]

        def role_nodes(role):
            edges = part[part["role"] == role][["exh_id", "cid"]]
            cids = np.sort(edges["cid"].unique())
            nodes = con.loc[cids].reset_index()
            node_of = pd.Series(np.arange(len(cids)), index=cids)
            edges = pd.DataFrame({"exh_id": edges["exh_id"].to_numpy(),
                                  role: node_of.loc[edges["cid"]].to_numpy()})
            return nodes, edges.sort_values(["exh_id", role]).reset_index(drop=True)

        artists, artist_edges = role_nodes("artist")
        curators, curator_edges = role_nodes("curator")
        return cls(ex, artists, curators, artist_edges, curator_edges)

    @property
    def num_artists(self):
        return len(self.artists)

    @property
    def num_curators(self):
        return len(self.curators)

    @property
    def num_exhibitions(self):
        return len(self.exhibitions)

    def with_shuffled_curators(self, seed):
        rng = np.random.default_rng(seed)
        edges = self.curator_edges.copy()
        decade = (self.exhibitions.loc[edges["exh_id"], "begin"].dt.year // 10).to_numpy()
        exh = edges["exh_id"].to_numpy().copy()
        for d in np.unique(decade):
            idx = np.flatnonzero(decade == d)
            exh[idx] = exh[rng.permutation(idx)]
        edges["exh_id"] = exh
        # A permutation can rarely pair a curator with the same exhibition twice.
        edges = edges.drop_duplicates().sort_values(["exh_id", "curator"]).reset_index(drop=True)
        return replace(self, curator_edges=edges)

## Core: temporal snapshots

A snapshot fixes a history cutoff and a set of target exhibitions after it. A model may see:

- every artist and curator edge of exhibitions that began before `history_end`;
- the curator edges and metadata of the target exhibitions.

It must rank the `candidates` (artists with at least one exhibition in the history) for each
target. The targets' artist edges are labels only.

For a cutoff year T:

| split | history | targets |
|---|---|---|
| train (rolling) | before t, for each t < T−5 | [t, min(t+5, T−5)) |
| validation | before T−5 | [T−5, T) |
| test | before T | [T, T+10) |

Rolling training snapshots have the same form as the test task, and the model never sees edges
from after its own targets.

In [ ]:
@dataclass
class Snapshot:
    history_end: pd.Timestamp
    targets: np.ndarray         # exh_ids, chronological
    hist_artist: np.ndarray     # [2, E] (exh_id, artist) for history exhibitions
    hist_curator: np.ndarray    # [2, E] (exh_id, curator) for history exhibitions
    target_curator: np.ndarray  # [2, E] (exh_id, curator) for target exhibitions
    candidates: np.ndarray      # artist ids with >=1 history exhibition, sorted
    positives: dict             # target exh_id -> all participating artist ids
    has_curator: np.ndarray     # per target: recorded curator in the original data

    @property
    def ref_year(self):
        ts = self.history_end
        return ts.year + (ts.day_of_year - 1) / 365.25

    @property
    def curator_edges(self):
        return np.concatenate([self.hist_curator, self.target_curator], axis=1)

    def label_matrix(self):
        """Boolean [num_targets, num_candidates]; True where the candidate exhibited."""
        labels = np.zeros((len(self.targets), len(self.candidates)), dtype=bool)
        for i, e in enumerate(self.targets):
            labels[i] = np.isin(self.candidates, self.positives[e], assume_unique=True)
        return labels


def _edges(df, col, mask):
    sub = df[mask]
    return np.stack([sub["exh_id"].to_numpy(), sub[col].to_numpy()]).astype(np.int64)


def make_snapshot(data, target_start, target_end, history_end=None):
    target_start, target_end = pd.Timestamp(target_start), pd.Timestamp(target_end)
    history_end = pd.Timestamp(history_end) if history_end is not None else target_start
    assert history_end <= target_start

    ex = data.exhibitions
    hist_ids = ex.index[ex["begin"] < history_end].to_numpy()
    targets = ex.index[(ex["begin"] >= target_start) & (ex["begin"] < target_end)].to_numpy().copy()

    ae, ce = data.artist_edges, data.curator_edges
    hist_artist = _edges(ae, "artist", ae["exh_id"].isin(hist_ids))
    hist_curator = _edges(ce, "curator", ce["exh_id"].isin(hist_ids))
    target_curator = _edges(ce, "curator", ce["exh_id"].isin(targets))

    target_artists = ae[ae["exh_id"].isin(targets)].groupby("exh_id")["artist"]
    positives = {e: np.array([], dtype=np.int64) for e in targets}
    positives.update({e: g.to_numpy() for e, g in target_artists})

    return Snapshot(
        history_end=history_end,
        targets=targets,
        hist_artist=hist_artist,
        hist_curator=hist_curator,
        target_curator=target_curator,
        candidates=np.unique(hist_artist[1]),
        positives=positives,
        has_curator=ex.loc[targets, "has_curator"].to_numpy(),
    )


def year_start(year):
    return pd.Timestamp(year=int(year), month=1, day=1)


def eval_snapshots(data, cutoff, split_cfg):
    val_start = cutoff - split_cfg["val_years"]
    val = make_snapshot(data, year_start(val_start), year_start(cutoff))
    test = make_snapshot(data, year_start(cutoff), year_start(cutoff + split_cfg["test_years"]))
    return val, test


def train_snapshots(data, cutoff, split_cfg):
    val_start = cutoff - split_cfg["val_years"]
    snaps = []
    for t in range(split_cfg["train_start"], val_start):
        end = min(t + split_cfg["train_window"], val_start)
        snap = make_snapshot(data, year_start(t), year_start(end))
        if len(snap.targets) and len(snap.candidates):
            snaps.append(snap)
    return snaps

## Core: metrics and evaluation protocol

Every model is a scorer with `score(snapshot) -> [num_targets, num_candidates]`, so candidate
pools, tie-breaking and metrics are identical across baselines and GNNs. Ties are broken at
random, not by candidate order.

The **primary subset** is test exhibitions that have a recorded curator and at least one returning
artist. Exhibitions whose artists are all MoMA debuts are kept with `n_pos_in_pool = 0` for the
cold-start analysis.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score


def rank_order(scores, rng):
    """Indices sorted by descending score, ties broken uniformly at random."""
    return np.lexsort((rng.random(len(scores)), -scores))


def ranking_metrics(scores, labels, ks, rng):
    """Per-query metrics for one exhibition. `labels` is a boolean vector with >=1 True."""
    order = rank_order(scores, rng)
    rel = labels[order].astype(float)
    n_pos = rel.sum()
    hits = np.cumsum(rel)
    precision_at = hits / np.arange(1, len(rel) + 1)
    discounts = 1.0 / np.log2(np.arange(2, len(rel) + 2))

    out = {}
    for k in ks:
        out[f"recall@{k}"] = hits[min(k, len(rel)) - 1] / n_pos
        out[f"map@{k}"] = (precision_at[:k] * rel[:k]).sum() / min(n_pos, k)
        ideal = discounts[:int(min(n_pos, k))].sum()
        out[f"ndcg@{k}"] = (discounts[:k] * rel[:k]).sum() / ideal
    if n_pos < len(rel):
        out["auc"] = roc_auc_score(labels, scores)
        out["ap"] = average_precision_score(labels, scores)
    else:
        out["auc"] = out["ap"] = np.nan

    ranks = np.empty(len(rel), dtype=np.int64)
    ranks[order] = np.arange(1, len(rel) + 1)
    return out, ranks[labels]


def evaluate(scorer, snap, ks, seed=0):
    """Returns (per-exhibition metrics, per-positive ranks)."""
    rng = np.random.default_rng(seed)
    scores = np.asarray(scorer.score(snap), dtype=np.float64)
    assert scores.shape == (len(snap.targets), len(snap.candidates))
    labels = snap.label_matrix()

    rows, rank_rows = [], []
    for i, e in enumerate(snap.targets):
        n_in_pool = int(labels[i].sum())
        row = {"exh_id": e, "has_curator": bool(snap.has_curator[i]),
               "n_pos": len(snap.positives[e]), "n_pos_in_pool": n_in_pool,
               "n_candidates": len(snap.candidates)}
        if n_in_pool:
            metrics, ranks = ranking_metrics(scores[i], labels[i], ks, rng)
            row.update(metrics)
            rank_rows.append(pd.DataFrame({
                "exh_id": e, "artist": snap.candidates[labels[i]], "rank": ranks,
                "n_candidates": len(snap.candidates)}))
        rows.append(row)
    per_exh = pd.DataFrame(rows)
    ranks = pd.concat(rank_rows, ignore_index=True) if rank_rows else pd.DataFrame()
    return per_exh, ranks


def metric_columns(per_exh):
    return [c for c in per_exh.columns if "@" in c or c in ("auc", "ap")]


def summarize(per_exh):
    """Mean metrics on the curated subset and on all exhibitions with >=1 returning artist."""
    scored = per_exh[per_exh["n_pos_in_pool"] > 0]
    cols = metric_columns(per_exh)
    return {name: {"n_exhibitions": len(subset), **subset[cols].mean().to_dict()}
            for name, subset in [("curated", scored[scored["has_curator"]]), ("all", scored)]}


def paired_bootstrap(a, b, n_boot=1000, seed=0, ci=0.95):
    """CI for mean(a - b), resampling the paired units (test exhibitions)."""
    diff = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    diff = diff[~np.isnan(diff)]
    rng = np.random.default_rng(seed)
    means = diff[rng.integers(0, len(diff), size=(n_boot, len(diff)))].mean(axis=1)
    lo, hi = np.quantile(means, [(1 - ci) / 2, 1 - (1 - ci) / 2])
    return {"mean_diff": diff.mean(), "ci_low": lo, "ci_high": hi, "n": len(diff),
            "p_le_0": float((means <= 0).mean())}


def load_per_exh(tag, cutoff):
    """Saved per-exhibition results (averaged over seeds) on the curated subset."""
    files = sorted((RESULTS_DIR / "gnn").glob(f"{tag}_T{cutoff}_s*_per_exh.csv"))
    files = files or sorted((RESULTS_DIR / "baselines").glob(f"{tag}_T{cutoff}_per_exh.csv"))
    if not files:
        return None, 0
    runs = pd.concat([pd.read_csv(f) for f in files])
    scored = runs[(runs["n_pos_in_pool"] > 0) & runs["has_curator"]]
    return scored.groupby("exh_id").mean(numeric_only=True), len(files)


def compare_runs(a, b, metric="map@20", cutoffs=None, n_boot=1000):
    """Paired bootstrap of A - B over curated test exhibitions, per cutoff.

    Tags: baselines by name (e.g. "curator_collab"); GNNs as "{model}_{mode}" (e.g. "rgcn_none").
    """
    rows = []
    for cutoff in cutoffs or CFG["split"]["cutoffs"]:
        (ra, na), (rb, nb) = load_per_exh(a, cutoff), load_per_exh(b, cutoff)
        if ra is None or rb is None:
            continue
        joined = ra[[metric]].merge(rb[[metric]], left_index=True, right_index=True,
                                    suffixes=("_a", "_b"))
        res = paired_bootstrap(joined[f"{metric}_a"], joined[f"{metric}_b"], n_boot=n_boot)
        rows.append({"cutoff": cutoff, "seeds_a": na, "seeds_b": nb,
                     a: joined[f"{metric}_a"].mean(), b: joined[f"{metric}_b"].mean(), **res})
    return pd.DataFrame(rows)

## Load data

In [ ]:
tables = load_tables()
data = MomaData.from_tables(tables)
print(f"{data.num_exhibitions} exhibitions, {data.num_artists} artists, "
      f"{data.num_curators} curators, {len(data.artist_edges)} artist edges, "
      f"{len(data.curator_edges)} curator edges")

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv, HeteroConv, HGTConv, RGCNConv, SAGEConv

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## Node features

- **Static** features are demographics and exhibition metadata: year, duration, and a MiniLM
  title embedding reduced with PCA.
- **History** features are exhibition count and years since the first and last exhibition. They
  are recomputed per snapshot from its visible history only.
- Exhibition features deliberately exclude curator counts, so the `none` ablation cannot recover
  curator information through them.

In [ ]:
YEAR_MIN, YEAR_SPAN = 1929, 60.0


def one_hot(values, categories):
    cat = pd.Categorical(values, categories=categories)
    out = np.zeros((len(values), len(categories)), dtype=np.float32)
    known = cat.codes >= 0
    out[np.flatnonzero(known), cat.codes[known]] = 1
    return out


def title_embeddings(titles, method, dim, seed=0):
    cache = PROCESSED_DIR / f"title_emb_{method}_{dim}.npy"
    if cache.exists():
        return np.load(cache)
    titles = [t if isinstance(t, str) else "" for t in titles]
    if method == "minilm":
        from sentence_transformers import SentenceTransformer
        from sklearn.decomposition import PCA
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        full = model.encode(titles, batch_size=128, show_progress_bar=False)
        emb = PCA(n_components=dim, random_state=seed).fit_transform(full)
    elif method == "tfidf":
        from sklearn.decomposition import TruncatedSVD
        from sklearn.feature_extraction.text import TfidfVectorizer
        tfidf = TfidfVectorizer(min_df=2, ngram_range=(1, 2)).fit_transform(titles)
        emb = TruncatedSVD(n_components=dim, random_state=seed).fit_transform(tfidf)
    else:
        raise ValueError(method)
    emb = (emb / (emb.std(axis=0, keepdims=True) + 1e-8)).astype(np.float32)
    np.save(cache, emb)
    return emb


def exhibition_features(data, fcfg):
    ex = data.exhibitions
    duration = ex["duration_days"]
    log_dur = np.log1p(duration.fillna(duration.median())).to_numpy()
    base = np.stack([
        (ex["year"].to_numpy() - YEAR_MIN) / YEAR_SPAN,
        (log_dur - log_dur.mean()) / log_dur.std(),
        duration.isna().to_numpy().astype(float),
    ], axis=1).astype(np.float32)
    titles = title_embeddings(ex["title"].tolist(), fcfg["title_encoder"], fcfg["title_dim"])
    return np.concatenate([base, titles], axis=1)


def person_static_features(nodes, top_nationalities):
    nat = nodes["nationality"]
    top = nat[nat != "unknown"].value_counts().index[:top_nationalities].tolist()
    nat_bucket = np.where(nat.isin(top), nat, np.where(nat == "unknown", "unknown", "other"))
    birth = nodes["birth_year"]
    return np.concatenate([
        one_hot(nodes["gender"], ["male", "female", "unknown"]),
        one_hot(nat_bucket, top + ["other", "unknown"]),
        np.stack([
            ((birth.fillna(1900) - 1900) / 50.0).to_numpy(),
            birth.isna().to_numpy().astype(float),
            (nodes["type"] == "Institution").to_numpy().astype(float),
        ], axis=1),
    ], axis=1).astype(np.float32)


def history_features(edges, num_nodes, exh_years, ref_year):
    """[num_nodes, 4]: log count, years since first / last exhibition (decades), has-history."""
    years = exh_years[edges[0]]
    count = np.bincount(edges[1], minlength=num_nodes)
    first = np.full(num_nodes, np.inf)
    last = np.full(num_nodes, -np.inf)
    np.minimum.at(first, edges[1], years)
    np.maximum.at(last, edges[1], years)
    seen = count > 0
    return np.stack([
        np.log1p(count),
        np.where(seen, (ref_year - first) / 10.0, 0.0),
        np.where(seen, (ref_year - last) / 10.0, 0.0),
        seen.astype(float),
    ], axis=1).astype(np.float32)


class FeatureStore:
    def __init__(self, data, cfg):
        fcfg = cfg["features"]
        self.data = data
        self.exh_years = data.exhibitions["year"].to_numpy()
        self.exhibition = exhibition_features(data, fcfg)
        self.artist_static = person_static_features(data.artists, fcfg["top_nationalities"])
        self.curator_static = person_static_features(data.curators, fcfg["top_nationalities"])

    def artist(self, snap):
        hist = history_features(snap.hist_artist, self.data.num_artists, self.exh_years,
                                snap.ref_year)
        return np.concatenate([self.artist_static, hist], axis=1)

    def curator(self, snap):
        hist = history_features(snap.hist_curator, self.data.num_curators, self.exh_years,
                                snap.ref_year)
        return np.concatenate([self.curator_static, hist], axis=1)


feats = FeatureStore(data, CFG)
print("exhibition features:", feats.exhibition.shape, " artist static:", feats.artist_static.shape)

## Graph construction

Every graph contains all nodes in the dataset; only the edges are snapshot-specific. Node ids
therefore stay stable, and nodes without visible history are isolated. `leak=True` exposes target
labels to message passing. It is used only for the sanity check below.

In [ ]:
ARTIST, EXHIBITION, CURATOR = "artist", "exhibition", "curator"
PARTICIPATES = (ARTIST, "participates", EXHIBITION)
HAS_ARTIST = (EXHIBITION, "has_artist", ARTIST)
CURATES = (CURATOR, "curates", EXHIBITION)
CURATED_BY = (EXHIBITION, "curated_by", CURATOR)


def edge_types(include_curators):
    return [PARTICIPATES, HAS_ARTIST] + ([CURATES, CURATED_BY] if include_curators else [])


def _exhibition_x(snap, feats):
    rel_year = (feats.exh_years - snap.ref_year)[:, None] / 10.0
    return torch.from_numpy(np.concatenate([feats.exhibition, rel_year], axis=1).astype(np.float32))


def _artist_edges(snap, leak):
    edges = snap.hist_artist
    if leak:
        leaked = [np.stack([np.full(len(a), e), a]) for e, a in snap.positives.items()]
        edges = np.concatenate([edges] + leaked, axis=1)
    return torch.from_numpy(edges)


def build_hetero(snap, feats, include_curators=True, leak=False):
    g = HeteroData()
    g[ARTIST].x = torch.from_numpy(feats.artist(snap))
    g[EXHIBITION].x = _exhibition_x(snap, feats)
    exh_artist = _artist_edges(snap, leak)
    g[PARTICIPATES].edge_index = exh_artist.flip(0)
    g[HAS_ARTIST].edge_index = exh_artist
    if include_curators:
        g[CURATOR].x = torch.from_numpy(feats.curator(snap))
        exh_curator = torch.from_numpy(snap.curator_edges)
        g[CURATES].edge_index = exh_curator.flip(0)
        g[CURATED_BY].edge_index = exh_curator
    return g


def coexhibition_edges(hist_artist, max_group=None, rng=None):
    """Artist-artist edges (both directions) between artists who shared an exhibition.
    `max_group` optionally subsamples very large group shows, whose cliques dominate."""
    order = np.argsort(hist_artist[0], kind="stable")
    exh, artists = hist_artist[0][order], hist_artist[1][order]
    src, dst = [], []
    for group in np.split(artists, np.flatnonzero(np.diff(exh)) + 1):
        if max_group is not None and len(group) > max_group:
            group = rng.choice(group, max_group, replace=False)
        if len(group) < 2:
            continue
        a, b = np.meshgrid(group, group)
        mask = a != b
        src.append(a[mask])
        dst.append(b[mask])
    if not src:
        return np.zeros((2, 0), dtype=np.int64)
    return np.unique(np.stack([np.concatenate(src), np.concatenate(dst)]), axis=1).astype(np.int64)


def build_homogeneous(snap, feats, max_group=None, seed=0, leak=False):
    g = HeteroData()
    g[ARTIST].x = torch.from_numpy(feats.artist(snap))
    g[EXHIBITION].x = _exhibition_x(snap, feats)
    edges = coexhibition_edges(_artist_edges(snap, leak).numpy(), max_group,
                               np.random.default_rng(seed))
    g[ARTIST, "coexhibits", ARTIST].edge_index = torch.from_numpy(edges)
    return g

In [ ]:
# Leakage check on the 1960 test graph: target exhibitions carry curator edges but no artists,
# and artist history counts include only history exhibitions.
_, test_1960 = eval_snapshots(data, 1960, CFG["split"])
g = build_hetero(test_1960, feats)
assert set(g[HAS_ARTIST].edge_index[0].tolist()).isdisjoint(test_1960.targets.tolist())
assert set(test_1960.target_curator[0]) <= set(g[CURATED_BY].edge_index[0].tolist())
counts = torch.expm1(g[ARTIST].x[:, -4]).round().long().numpy()
assert np.array_equal(counts, np.bincount(test_1960.hist_artist[1], minlength=data.num_artists))

# Shuffle ablation preserves curator degrees (up to rare duplicate pairs) but moves most edges.
shuffled = data.with_shuffled_curators(seed=0)
before = data.curator_edges.groupby("curator").size()
after = shuffled.curator_edges.groupby("curator").size().reindex(before.index, fill_value=0)
assert (before - after).abs().sum() <= 0.02 * len(data.curator_edges)
assert len(data.curator_edges.merge(shuffled.curator_edges)) < 0.5 * len(data.curator_edges)
print("graph checks passed")
g

## Models

Each encoder maps a snapshot graph to artist and exhibition embeddings. The two-tower decoder
scores pairs as score(a, e) = ⟨f(h_a), g(h_e)⟩ + b(h_a). The artist bias lets the model represent
popularity directly. Scoring every candidate for every target is then a single matrix product.

In [ ]:
class TypeProjection(nn.Module):
    def __init__(self, in_dims, hidden):
        super().__init__()
        self.lins = nn.ModuleDict({t: nn.Linear(d, hidden) for t, d in in_dims.items()})

    def forward(self, x_dict):
        return {t: self.lins[t](x) for t, x in x_dict.items()}


class RGCNEncoder(nn.Module):
    def __init__(self, in_dims, edge_types, hidden, layers, dropout):
        super().__init__()
        self.node_types = list(in_dims)
        self.edge_types = list(edge_types)
        self.proj = TypeProjection(in_dims, hidden)
        self.convs = nn.ModuleList(
            RGCNConv(hidden, hidden, num_relations=len(self.edge_types)) for _ in range(layers))
        self.dropout = nn.Dropout(dropout)

    def forward(self, g):
        h_dict = self.proj(g.x_dict)
        offsets, start = {}, 0
        for t in self.node_types:
            offsets[t] = start
            start += h_dict[t].shape[0]
        x = torch.cat([h_dict[t] for t in self.node_types])

        edge_index, edge_type = [], []
        for rel, (src, name, dst) in enumerate(self.edge_types):
            ei = g[src, name, dst].edge_index
            edge_index.append(torch.stack([ei[0] + offsets[src], ei[1] + offsets[dst]]))
            edge_type.append(torch.full((ei.shape[1],), rel, dtype=torch.long, device=ei.device))
        edge_index, edge_type = torch.cat(edge_index, 1), torch.cat(edge_type)

        x = self.dropout(torch.relu(x))
        for conv in self.convs:
            x = x + self.dropout(torch.relu(conv(x, edge_index, edge_type)))
        out = {t: x[offsets[t]:offsets[t] + h_dict[t].shape[0]] for t in self.node_types}
        return out[ARTIST], out[EXHIBITION]


class HeteroSAGEEncoder(nn.Module):
    """HeteroConv with a separate GraphSAGE operator per relation."""

    def __init__(self, in_dims, edge_types, hidden, layers, dropout):
        super().__init__()
        self.proj = TypeProjection(in_dims, hidden)
        self.convs = nn.ModuleList(
            HeteroConv({et: SAGEConv(hidden, hidden) for et in edge_types}, aggr="sum")
            for _ in range(layers))
        self.dropout = nn.Dropout(dropout)

    def forward(self, g):
        h = {t: self.dropout(torch.relu(x)) for t, x in self.proj(g.x_dict).items()}
        for conv in self.convs:
            update = conv(h, g.edge_index_dict)
            h = {t: h[t] + self.dropout(torch.relu(update[t])) if t in update else h[t]
                 for t in h}
        return h[ARTIST], h[EXHIBITION]


class HGTEncoder(nn.Module):
    def __init__(self, in_dims, edge_types, hidden, layers, dropout, heads=2):
        super().__init__()
        metadata = (list(in_dims), list(edge_types))
        self.proj = TypeProjection(in_dims, hidden)
        self.convs = nn.ModuleList(HGTConv(hidden, hidden, metadata, heads=heads)
                                   for _ in range(layers))
        self.dropout = nn.Dropout(dropout)

    def forward(self, g):
        h = {t: self.dropout(torch.relu(x)) for t, x in self.proj(g.x_dict).items()}
        for conv in self.convs:
            update = conv(h, g.edge_index_dict)
            h = {t: h[t] + self.dropout(update[t]) if update.get(t) is not None else h[t]
                 for t in h}
        return h[ARTIST], h[EXHIBITION]


class HomogeneousEncoder(nn.Module):
    """GraphSAGE or GAT over the artist co-exhibition graph; MLP over exhibition metadata."""

    def __init__(self, in_dims, hidden, layers, dropout, conv="sage", heads=2):
        super().__init__()
        self.artist_in = nn.Linear(in_dims[ARTIST], hidden)
        if conv == "sage":
            make = lambda: SAGEConv(hidden, hidden)
        elif conv == "gat":
            make = lambda: GATConv(hidden, hidden // heads, heads=heads)
        else:
            raise ValueError(conv)
        self.convs = nn.ModuleList(make() for _ in range(layers))
        self.exhibition_mlp = nn.Sequential(
            nn.Linear(in_dims[EXHIBITION], hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden))
        self.dropout = nn.Dropout(dropout)

    def forward(self, g):
        edge_index = g[ARTIST, "coexhibits", ARTIST].edge_index
        x = self.dropout(torch.relu(self.artist_in(g[ARTIST].x)))
        for conv in self.convs:
            x = x + self.dropout(torch.relu(conv(x, edge_index)))
        return x, self.exhibition_mlp(g[EXHIBITION].x)


HETERO_MODELS = {"rgcn": RGCNEncoder, "heteroconv": HeteroSAGEEncoder, "hgt": HGTEncoder}
HOMOGENEOUS_MODELS = {"sage", "gat"}


class TwoTowerDecoder(nn.Module):
    def __init__(self, hidden, dropout):
        super().__init__()

        def mlp():
            return nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden))

        self.artist, self.exhibition = mlp(), mlp()
        self.bias = nn.Linear(hidden, 1)

    def forward(self, h_artist, h_exhibition):
        return self.exhibition(h_exhibition) @ self.artist(h_artist).T + self.bias(h_artist).T


class LinkModel(nn.Module):
    def __init__(self, name, in_dims, hidden, layers, dropout, include_curators=True):
        super().__init__()
        if name in HETERO_MODELS:
            self.encoder = HETERO_MODELS[name](in_dims, edge_types(include_curators), hidden,
                                               layers, dropout)
        elif name in HOMOGENEOUS_MODELS:
            self.encoder = HomogeneousEncoder(in_dims, hidden, layers, dropout, conv=name)
        else:
            raise ValueError(name)
        self.decoder = TwoTowerDecoder(hidden, dropout)

    def forward(self, g, targets, candidates):
        """Scores [len(targets), len(candidates)]."""
        h_artist, h_exhibition = self.encoder(g)
        return self.decoder(h_artist[candidates], h_exhibition[targets])

## Training

Each epoch takes one optimizer step per rolling training snapshot. The loss is BPR (or BCE) with
uniformly sampled negatives, and each target exhibition gets equal weight, as in the
per-exhibition evaluation. Model selection uses validation MAP@20 on curated exhibitions.

In [ ]:
@dataclass
class Prepared:
    graph: object
    targets: torch.Tensor
    candidates: torch.Tensor
    labels: torch.Tensor        # bool [targets, candidates]


class GraphBuilder:
    def __init__(self, feats, model_cfg, seed=0, leak=False):
        self.feats = feats
        self.name = model_cfg["name"]
        self.include_curators = model_cfg["curator_mode"] != "none"
        self.max_group = model_cfg.get("max_group")
        self.seed = seed
        self.leak = leak  # sanity check only: target labels visible to message passing

    def __call__(self, snap):
        if self.name in HOMOGENEOUS_MODELS:
            g = build_homogeneous(snap, self.feats, self.max_group, self.seed, leak=self.leak)
        else:
            g = build_hetero(snap, self.feats, self.include_curators, leak=self.leak)
        return g.to(DEVICE)

    def prepare(self, snap):
        return Prepared(self(snap), torch.from_numpy(snap.targets).to(DEVICE),
                        torch.from_numpy(snap.candidates).to(DEVICE),
                        torch.from_numpy(snap.label_matrix()).to(DEVICE))


class GNNScorer:
    """Adapts a trained LinkModel to the shared evaluation protocol."""

    def __init__(self, model, builder, name):
        self.model, self.builder, self.name = model, builder, name

    @torch.no_grad()
    def score(self, snap):
        self.model.eval()
        scores = self.model(self.builder(snap), torch.from_numpy(snap.targets).to(DEVICE),
                            torch.from_numpy(snap.candidates).to(DEVICE))
        return scores.cpu().numpy()


def ranking_loss(scores, labels, kind, num_neg, gen):
    t_idx, c_idx = labels.nonzero(as_tuple=True)
    if len(t_idx) == 0:
        return None
    neg = torch.randint(0, labels.shape[1], (len(t_idx), num_neg), generator=gen).to(labels.device)
    valid = (~labels[t_idx[:, None], neg]).float()
    pos_scores = scores[t_idx, c_idx]
    neg_scores = scores[t_idx[:, None], neg]
    if kind == "bpr":
        per_pair = -(F.logsigmoid(pos_scores[:, None] - neg_scores) * valid).sum(1)
    elif kind == "bce":
        per_pair = (F.binary_cross_entropy_with_logits(pos_scores, torch.ones_like(pos_scores),
                                                       reduction="none")[:, None]
                    + F.binary_cross_entropy_with_logits(neg_scores, torch.zeros_like(neg_scores),
                                                         reduction="none") * valid).sum(1)
    else:
        raise ValueError(kind)
    per_pair = per_pair / valid.sum(1).clamp(min=1)
    n_pos = labels.sum(1).float()
    return (per_pair / n_pos[t_idx]).sum() / (n_pos > 0).sum()


def train_model(data, feats, cutoff, cfg, seed, leak=False, verbose=True):
    """Train on snapshots before the validation window; select by validation MAP@K."""
    mcfg, ks = cfg["model"], cfg["eval"]["ks"]
    metric = cfg["eval"]["primary_metric"]
    torch.manual_seed(seed)
    np.random.seed(seed)
    gen = torch.Generator().manual_seed(seed)

    builder = GraphBuilder(feats, mcfg, seed, leak=leak)
    train = [builder.prepare(s) for s in train_snapshots(data, cutoff, cfg["split"])]
    val_snap, _ = eval_snapshots(data, cutoff, cfg["split"])

    in_dims = {t: train[0].graph[t].x.shape[1] for t in train[0].graph.node_types}
    model = LinkModel(mcfg["name"], in_dims, mcfg["hidden"], mcfg["layers"], mcfg["dropout"],
                      include_curators=builder.include_curators).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=mcfg["lr"], weight_decay=mcfg["weight_decay"])
    scorer = GNNScorer(model, builder, mcfg["name"])

    best = {"score": -np.inf, "epoch": 0, "state": None}
    history, stale = [], 0
    for epoch in range(1, mcfg["epochs"] + 1):
        model.train()
        losses = []
        for i in torch.randperm(len(train), generator=gen).tolist():
            p = train[i]
            loss = ranking_loss(model(p.graph, p.targets, p.candidates), p.labels,
                                mcfg["loss"], mcfg["num_neg"], gen)
            if loss is None:
                continue
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())

        if epoch % mcfg["eval_every"] == 0:
            per_exh, _ = evaluate(scorer, val_snap, ks, seed=seed)
            val = summarize(per_exh)["curated"][metric]
            history.append({"epoch": epoch, "loss": float(np.mean(losses)), f"val_{metric}": val})
            if verbose:
                print(f"  epoch {epoch:4d}  loss {np.mean(losses):.4f}  val {metric} {val:.4f}")
            if val > best["score"]:
                best = {"score": val, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}
                stale = 0
            else:
                stale += 1
                if stale >= mcfg["patience"]:
                    break

    model.load_state_dict(best["state"])
    return model, builder, {"best_epoch": best["epoch"], f"best_val_{metric}": best["score"],
                            "history": history}


def run_experiment(model_name, curator_mode, cutoff, seed, cfg=CFG, verbose=False):
    """Train one model, evaluate on the test window, save per-exhibition results."""
    cfg = copy.deepcopy(cfg)
    cfg["model"].update(name=model_name, curator_mode=curator_mode)
    run_data = data.with_shuffled_curators(seed) if curator_mode == "shuffled" else data
    tag = f"{model_name}_{curator_mode}"
    start = time.time()
    model, builder, info = train_model(run_data, feats, cutoff, cfg, seed, verbose=verbose)
    _, test = eval_snapshots(run_data, cutoff, cfg["split"])
    per_exh, ranks = evaluate(GNNScorer(model, builder, tag), test, cfg["eval"]["ks"], seed=seed)

    out_dir = RESULTS_DIR / "gnn"
    out_dir.mkdir(exist_ok=True)
    run = f"{tag}_T{cutoff}_s{seed}"
    per_exh.to_csv(out_dir / f"{run}_per_exh.csv", index=False)
    ranks.to_csv(out_dir / f"{run}_ranks.csv", index=False)
    summary = summarize(per_exh)
    with open(out_dir / f"{run}_info.json", "w") as f:
        json.dump({"model": cfg["model"], "summary": summary, **info}, f, indent=2, default=float)
    print(f"{run:28s} best epoch {info['best_epoch']:3d}  "
          f"test map@20 {summary['curated']['map@20']:.4f}  "
          f"recall@50 {summary['curated']['recall@50']:.4f}  ({time.time() - start:.0f}s)")
    return [{"model": tag, "cutoff": cutoff, "seed": seed, "subset": s,
             "best_epoch": info["best_epoch"], **summary[s]} for s in summary]

## Sanity check: label leakage

This trains a second model with target labels visible to message passing, during both training
and testing. It should score far above the masked model: on the 1960 cutoff it reaches AUC
≈ 0.99 and MAP@20 ≈ 0.24, against ≈ 0.93 and ≈ 0.09 masked. The masked model's lower scores
therefore come from the masking, not from a model that cannot use the graph.

In [ ]:
RUN_LEAK_CHECK = True
if RUN_LEAK_CHECK:
    leak_model, leak_builder, _ = train_model(data, feats, 1960, CFG, seed=0, leak=True,
                                              verbose=False)
    leaky, _ = evaluate(GNNScorer(leak_model, leak_builder, "leak"), test_1960, CFG["eval"]["ks"])
    print({k: round(v, 3) for k, v in summarize(leaky)["curated"].items()})

## Experiments

- **Main ablation**: R-GCN with `full`, `none` and `shuffled` curators.
- **Exhibition-node ablation**: homogeneous `sage` / `gat` vs `rgcn_none`.

Use `SEEDS = CFG["seeds"]` (5 seeds) for final numbers. A single seed is enough for a quick look.

In [ ]:
EXPERIMENTS = [("rgcn", "full"), ("rgcn", "none"), ("rgcn", "shuffled")]
# Final report: add ("heteroconv", "full"), ("hgt", "full"), ("sage", "none"), ("gat", "none").
CUTOFFS = CFG["split"]["cutoffs"]
SEEDS = [0]

rows = []
for model_name, mode in EXPERIMENTS:
    for cutoff in CUTOFFS:
        for seed in SEEDS:
            rows += run_experiment(model_name, mode, cutoff, seed)
gnn_table = pd.DataFrame(rows)
gnn_table.to_csv(RESULTS_DIR / "gnn" / "summary.csv", index=False)

## Results

In [ ]:
cols = ["recall@10", "recall@50", "map@20", "ndcg@20", "auc"]
curated = gnn_table[gnn_table["subset"] == "curated"]
baseline_file = RESULTS_DIR / "baselines" / "summary.csv"
if baseline_file.exists():
    base = pd.read_csv(baseline_file)
    curated = pd.concat([base[base["subset"] == "curated"], curated])
curated.pivot_table(index="model", columns="cutoff", values="map@20").round(4)

In [ ]:
# Mean (over seeds and cutoffs) on the curated subset.
curated.groupby("model")[cols].mean().sort_values("map@20", ascending=False).round(4)

### Curator ablation: paired bootstrap over test exhibitions

- **full − none** tests the overall value of curator information.
- **full − shuffled** tests whether the *specific* historical curator assignments matter beyond
  the extra connectivity.

In [ ]:
for a, b in [("rgcn_full", "rgcn_none"), ("rgcn_full", "rgcn_shuffled"),
             ("rgcn_full", "curator_collab"), ("rgcn_none", "popularity")]:
    table = compare_runs(a, b, metric="map@20")
    if len(table):
        print(f"map@20: {a} − {b}")
        display(table.round(4))